# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets and fields, referencing their `@id` values. This enables targeted data extraction and ensures reproducibility.

In [ ]:
# List all record sets in the dataset with their @id and details
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in the metadata. Please check the dataset or schema definition.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', '-')}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    - {field['@id']}", end='')
            name = field.get('name', None)
            if name:
                print(f" (name: {name})")
            else:
                print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Identify the record set and field `@id`s from the overview above.

In [ ]:
# Replace the record set and field @id's below with those discovered in the overview step for your exploration
# For demonstration, we'll programmatically fetch them

record_sets_ids = []
# Extract all record set @id's
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        record_sets_ids.append(rs['@id'])
else:
    raise RuntimeError("No record sets found in the metadata.")

dataframes = {}
for rset_id in record_sets_ids:
    # Load all records from this record set
    try:
        records = list(dataset.records(record_set=rset_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rset_id] = df
            print(f"Loaded records for record set {rset_id}, columns:\n  {df.columns.tolist()}")
            print(df.head(2))
        else:
            print(f"No records found for {rset_id}.")
    except Exception as e:
        print(f"Failed to read from {rset_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section shows filtering, normalization, and grouping operations.

In [ ]:
# Choose a record set for EDA
if not dataframes:
    raise RuntimeError("No dataframes found for EDA analysis. Check data extraction step.")

# Pick the first loaded record set for demonstration
record_set_id = list(dataframes.keys())[0]
df = dataframes[record_set_id]

# List numeric fields
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_fields:
    print("No numeric fields found in this record set.")
else:
    numeric_field = numeric_fields[0]  # Just pick the first one for demo
    print(f"Using numeric field for analysis: {numeric_field}")

    threshold = df[numeric_field].mean()  # Use mean as a demo threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):\n{filtered_df.head()}")

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a non-numeric field
    group_fields = df.select_dtypes(include=['object']).columns.tolist()
    if group_fields:
        group_field = group_fields[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean values):\n{grouped_df.head()}")
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if we have a numeric field
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If grouping, plot mean by group field
    if 'group_field' in locals():
        mean_by_group = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(data=mean_by_group, x=group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

- Metadata and available record sets were reviewed using their `@id` values.
- Data was extracted and loaded into DataFrames for analysis.
- Simple exploratory data analysis and visualization were demonstrated, including filtering and normalizing numeric fields.

To go further:
- Review field descriptions and experiment with different fields and record sets.
- Apply advanced filtering, reshaping, or domain-specific analyses.